# PPO agents

> PPO based agent

In [ ]:
#| default_exp agents.rl.RL2ppo

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import logging

# set logging level to INFO
logging.basicConfig(level=logging.INFO)

from abc import ABC, abstractmethod
from typing import Union, Optional, List, Tuple
import numpy as np
import os

from ddopai.envs.base import BaseEnvironment
from ddopai.agents.rl.mushroom_rl import MushroomBaseAgent
from ddopai.utils import MDPInfo, Parameter
from ddopai.agents.obsprocessors import FlattenTimeDimNumpy
from ddopai.RL_approximators import MLPState, MLPActor, RL2RNNActor, RL2RNNValue
from ddopai.envs.actionprocessors import ClipAction

from ddopai.dataloaders.base import BaseDataLoader
from mushroom_rl.policy import GaussianTorchPolicy

from mushroom_rl.core import Agent
from mushroom_rl.approximators import Regressor
from mushroom_rl.approximators.parametric import TorchApproximator
from mushroom_rl.utils.torch import to_float_tensor, update_optimizer_parameters
from mushroom_rl.utils.minibatches import minibatch_generator
from mushroom_rl.utils.dataset import parse_dataset, compute_J
from mushroom_rl.utils.value_functions import compute_gae
from mushroom_rl.utils.parameters import to_parameter

import torch
import torch.optim as optim
import torch.nn.functional as F
from torchinfo import summary

import time

In [ ]:
# export

class RL2PPO(Agent):
    """
    Proximal Policy Optimization (PPO) Agent supporting sequential data and RL² compatibility.
    """

    def __init__(self, mdp_info, policy, actor_optimizer, critic_params,
                 n_epochs_policy, batch_size, eps_ppo, lam, ent_coeff=0.0,
                 critic_fit_params=None):
        """
        Constructor.

        Args:
            mdp_info (MDPInfo): Environment information.
            policy (TorchPolicy): Actor network.
            actor_optimizer (dict): Optimizer settings for actor.
            critic_params (dict): Parameters for critic network.
            n_epochs_policy (int or Parameter): PPO epochs per fit call.
            batch_size (int or Parameter): Batch size for PPO update.
            eps_ppo (float or Parameter): PPO clipping parameter.
            lam (float or Parameter): Lambda for GAE computation.
            ent_coeff (float or Parameter): Entropy regularization coefficient.
            critic_fit_params (dict): Optional parameters for fitting the critic.
        """
        # Save parameters
        self._critic_fit_params = dict(n_epochs=10) if critic_fit_params is None else critic_fit_params
        self._n_epochs_policy = to_parameter(n_epochs_policy)
        self._batch_size = to_parameter(batch_size)
        self._eps_ppo = to_parameter(eps_ppo)
        self._lambda = to_parameter(lam)
        self._ent_coeff = to_parameter(ent_coeff)

        # Build optimizer
        self._optimizer = actor_optimizer['class'](policy.parameters(), **actor_optimizer['params'])

        # Build value function approximator
        self._V = Regressor(TorchApproximator, **critic_params)
        # Buffer for storing log-probabilities and value predictions
        self._vpreds_buffer = []
        self._logpacs_buffer = []

        # Iteration counter
        self._iter = 1

        # Save attributes for checkpointing
        self._add_save_attr(
            _critic_fit_params='pickle',
            _n_epochs_policy='mushroom',
            _batch_size='mushroom',
            _eps_ppo='mushroom',
            _lambda='mushroom',
            _ent_coeff='mushroom',
            _optimizer='torch',
            _V='mushroom',
            _iter='primitive'
        )

        super().__init__(mdp_info, policy, None)

    def draw_action(self, state):
        """
        Overriding draw_action to also save logpacs and vpreds during rollout.
        """
        # Standard draw
        action = super().draw_action(state)

        # Save logpac and vpred
        with torch.no_grad():
            obs_tensor = to_float_tensor(np.expand_dims(state, axis=0), self.policy.use_cuda)

            vpred = self._V(obs_tensor).squeeze(0)
            pol_dist = self.policy.distribution_t(obs_tensor)
            # TODO: Check if i need to do [:, None].detach() instead
            logpac = pol_dist.log_prob(to_float_tensor(action, self.policy.use_cuda))

            self._vpreds_buffer.append(vpred.cpu().numpy())
            self._logpacs_buffer.append(logpac.cpu().numpy())

        return action
    
    def fit(self, dataset, **info):
        """
        Fit the policy and value networks using PPO loss.

        Args:
            dataset (list): Dataset collected from environment.
            logpacs (Tensor or None): Log-probabilities of actions at rollout.
            vpreds (Tensor or None): Value predictions at rollout.
        """
        # Use the saved buffers
        vpreds = np.stack(self._vpreds_buffer)
        logpacs = np.stack(self._logpacs_buffer)

        # Clear the buffers
        self._vpreds_buffer = []
        self._logpacs_buffer = []
        
        # Parse dataset
        x, u, r, xn, absorbing, last = parse_dataset(dataset)
        x = x.astype(np.float32)
        u = u.astype(np.float32)
        r = r.astype(np.float32)
        xn = xn.astype(np.float32)

        obs = to_float_tensor(x, self.policy.use_cuda)
        act = to_float_tensor(u, self.policy.use_cuda)

        # 1. Compute v_targets and advantages
        if vpreds is None:
            # Compute vpreds from current critic if not provided
            v_target, np_adv = compute_gae(self._V, x, xn, r, absorbing, last,
                                           self.mdp_info.gamma, self._lambda())
        else:
            # Otherwise assume vpreds are already stored
            v_target, np_adv = self._compute_gae_from_vpreds(vpreds, r, absorbing, last)

        np_adv = (np_adv - np.mean(np_adv)) / (np.std(np_adv) + 1e-8)
        adv = to_float_tensor(np_adv, self.policy.use_cuda)

        if logpacs is None:
            # Recompute old log-probs if not provided
            old_pol_dist = self.policy.distribution_t(obs)
            old_log_p = old_pol_dist.log_prob(act)[:, None].detach()
        else:
            # Use stored logpacs
            old_log_p = logpacs

        # 2. Fit critic
        self._V.fit(x, v_target, **self._critic_fit_params)

        # 3. Update actor
        self._update_policy(obs, act, adv, old_log_p)

        self._iter += 1

    def _compute_gae_from_vpreds(self, vpreds, rewards, absorbing, last):
        gamma = self.mdp_info.gamma
        lam = self._lambda()

        vpreds_next = np.concatenate([vpreds[1:], np.array([0.0])])

        advs = np.empty_like(vpreds)
        for t in reversed(range(len(vpreds))):
            if last[t] or t == len(vpreds) - 1:
                next_non_terminal = 1.0 - absorbing[t]
                delta = rewards[t] - vpreds[t]
                if next_non_terminal:
                    delta += gamma * vpreds_next[t]
                advs[t] = delta
            else:
                next_non_terminal = 1.0 - absorbing[t]
                delta = rewards[t] + gamma * vpreds_next[t] - vpreds[t]
                advs[t] = delta + gamma * lam * advs[t+1]

        v_target = advs + vpreds
        return v_target, advs


    def _update_policy(self, obs, act, adv, old_log_p):
        """
        Perform PPO policy update.
        """
        for _ in range(self._n_epochs_policy()):
            for obs_i, act_i, adv_i, old_log_p_i in minibatch_generator(
                    self._batch_size(), obs, act, adv, old_log_p):
                self._optimizer.zero_grad()
                prob_ratio = torch.exp(
                    self.policy.log_prob_t(obs_i, act_i) - old_log_p_i
                )
                clipped_ratio = torch.clamp(prob_ratio, 1 - self._eps_ppo(),
                                         1 + self._eps_ppo())
                loss = -torch.mean(torch.min(prob_ratio * adv_i,
                                            clipped_ratio * adv_i))
                loss -= self._ent_coeff()*self.policy.entropy_t(obs_i)
                loss.backward()
                self._optimizer.step()

    def _post_load(self):
        if self._optimizer is not None:
            update_optimizer_parameters(self._optimizer, list(self.policy.parameters()))


In [ ]:
# | export

class RL2PPOAgent(MushroomBaseAgent):
    """
    RL² PPO Agent for meta-learning, based on recurrent policy/value networks and MushroomRL core agent.
    """

    def __init__(self,
                 environment_info: MDPInfo,
                 hidden_layers_RNN: int = 1,
                 num_hidden_units_RNN: int = 64,
                 hidden_layers_MLP: List = None,
                 activation: str = "relu",
                 learning_rate_actor: float = 3e-4,
                 learning_rate_critic: float | None = None,
                 batch_size: int = 64,
                 n_epochs_policy: int = 4,
                 eps_ppo: float = 0.2,
                 lam: float = 0.95,
                 ent_coeff: float = 0.0,
                 drop_prob: float = 0.0,
                 batch_norm: bool = False,
                 init_method: str = "xavier_uniform",
                 optimizer: str = "Adam",
                 loss: str = "MSE",
                 obsprocessors: list | None = None,
                 device: str = "cpu",
                 agent_name: str | None = "RL2PPO",
                 RNN_cell: str = "GRU",
                 std_0: float = 0.1, # Unused but kept for consistency
                 ):
        """
        Constructor. Sets up policy, critic, and PPO core agent.
        """

        # 1. Device setup
        self.n_steps_per_fit = None  # In RL² training loop, external control of fitting.
        use_cuda = self.set_device(device)

        # 2. Input shapes
        input_shape = self.get_input_shape(environment_info.observation_space)
        actor_output_shape = environment_info.action_space.shape
        input_shape = self.convert_recursively_to_int(input_shape)
        actor_output_shape = self.convert_recursively_to_int(actor_output_shape)

        # 3. Optimizers
        OptimizerClass = self.get_optimizer_class(optimizer)
        learning_rate_critic = learning_rate_critic or learning_rate_actor
        loss_function = self.get_loss_function(loss)

        # 4. Define actor network (RL² recurrent actor)
        hidden_layers_MLP = hidden_layers_MLP or [64, 64]
        
        policy_params = dict(
            network=RL2RNNActor,  # <== ⚡ RL² actor class
            input_shape=input_shape,
            output_shape=actor_output_shape,
            hidden_layers_RNN=hidden_layers_RNN,
            num_hidden_units_RNN=num_hidden_units_RNN,
            hidden_layers_MLP=hidden_layers_MLP,
            RNN_cell=RNN_cell,
            activation=activation,
            final_activation="identity",
            drop_prob=drop_prob,
            batch_norm=batch_norm,
            init_method=init_method,
            use_cuda=use_cuda,
            dropout=self.dropout,
        )

        policy = GaussianTorchPolicy(**policy_params)

        # 5. Define critic network (RL² recurrent value net)
        critic_params = dict(
            network=RL2RNNValue,  # <== ⚡ RL² critic class
            optimizer={'class': OptimizerClass, 'params': {'lr': learning_rate_critic}},
            loss=loss_function,
            input_shape=input_shape,
            output_shape=(1,),
            hidden_layers_RNN=hidden_layers_RNN,
            num_hidden_units_RNN=num_hidden_units_RNN,
            hidden_layers_MLP=hidden_layers_MLP,
            RNN_cell=RNN_cell,
            activation=activation,
            final_activation="identity",
            drop_prob=drop_prob,
            batch_norm=batch_norm,
            init_method=init_method,
            use_cuda=use_cuda,
            dropout=self.dropout,
        )

        actor_optimizer = {
            'class': OptimizerClass,
            'params': {'lr': learning_rate_actor}
        }

        # 6. Build the MushroomRL PPO core agent (with our RL² actor and critic)
        self.agent = RL2PPO(
            mdp_info=environment_info,
            policy=policy,
            actor_optimizer=actor_optimizer,
            critic_params=critic_params,
            n_epochs_policy=n_epochs_policy,
            batch_size=batch_size,
            eps_ppo=eps_ppo,
            lam=lam,
            ent_coeff=ent_coeff,
            critic_fit_params=None
        )

        # 7. Build the MushroomBaseAgent (super class)
        super().__init__(
            environment_info=environment_info,
            obsprocessors=obsprocessors,
            device=device,
            agent_name=agent_name
        )

        # 8. Logging networks
        logging.info("Actor (RL²) network:")
        if logging.getLogger().isEnabledFor(logging.INFO):
            input_size = self.add_batch_dimension_for_shape(input_shape)
            print(summary(self.actor, input_size=input_size))
            time.sleep(.2)

        logging.info("Critic (RL²) network:")
        if logging.getLogger().isEnabledFor(logging.INFO):
            input_size = self.add_batch_dimension_for_shape(input_shape)
            print(summary(self.critic, input_size=input_size))

    def reset_hidden(self, batch_size=1, device='cpu'):
        """
        Reset the hidden state of both policy (actor) and value (critic) networks.
        
        Args:
            batch_size (int): number of parallel episodes/tasks (usually 1).
            device (str): device where hidden states should be allocated ('cpu' or 'cuda').
        """
        # Reset actor hidden state
        actor_network = self.agent.policy._mu._impl.model.network
        actor_network.hidden_state = actor_network.model.rnn.model.init_hidden(batch_size=batch_size, device=device)

        # Reset critic hidden state
        critic_network = self.agent._V._impl.model.network
        critic_network.hidden_state = critic_network.model.rnn.model.init_hidden(batch_size=batch_size, device=device)

    def get_network_list(self, set_actor_critic_attributes: bool = True):
        """
        Returns the list of actor and critic networks for saving/loading.
        """
        critic = self.agent._V._impl.model.network
        actor = self.agent.policy._mu._impl.model.network

        networks = [critic, actor]

        if set_actor_critic_attributes:
            return networks, actor, critic
        else:
            return networks
